# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works
Sections 1–7 contain instructions to write your own code in the empty cell provided under each instruction.

Section 8 is pre-written for evaluator use and must not be modified. We will run it against a hidden test set to score your submission.

Your notebook must run top-to-bottom without errors before submission (**Restart Kernel & Run All**).


## 0. Setup
Standard imports, and file paths are set up below. Add any additional imports your approach needs in the same cell.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Additional models used for comparison
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# File paths
TRAIN_PATH = "/content/dataset-1.csv"
TARGET_COLUMN = "Churn"
EVALUATION_SPLIT_PATH = "./evaluation_split.csv"  # organizers only — do not rely on this file existing locally


## 1. Load the Data
Load the dataset from `TRAIN_PATH` into a dataframe.


In [ ]:
df = pd.read_csv(TRAIN_PATH)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (5634, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False


## 2. Explore and Clean the Dataset
Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and outliers.
- Document any rows/columns you drop and why.


In [ ]:
def data_cleaning(df):
    df = df.copy()

    # Remove exact duplicate rows if any are present.
    df = df.drop_duplicates().reset_index(drop=True)

    # customerID is only an identifier and should not be used to predict churn.
    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])

    # TotalCharges should be numeric. Blank strings become NaN and are later
    # imputed by the preprocessing pipeline.
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Remove accidental leading/trailing spaces from categorical values.
    object_columns = df.select_dtypes(include=["object"]).columns
    for col in object_columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    return df

# Inspect the result of cleaning without overwriting the original raw dataframe.
cleaned_preview = data_cleaning(df)
print("Shape after cleaning:", cleaned_preview.shape)
print("Missing values after cleaning:")
display(cleaned_preview.isnull().sum()[cleaned_preview.isnull().sum() > 0])


Shape after cleaning: (5634, 20)
Missing values after cleaning:


,0
TotalCharges,8


## 3. Feature Engineering
Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, extracting information from dates or creating new columns from existing ones.


In [ ]:
def feature_engineering(df):
    df = df.copy()

    # Average monthly spend across the customer's lifetime.
    # The +1 prevents division by zero for new customers with tenure = 0.
    if {"TotalCharges", "tenure"}.issubset(df.columns):
        df["AvgChargesPerMonth"] = df["TotalCharges"] / (df["tenure"] + 1)

    # Count how many optional service-related products the customer uses.
    service_columns = [
        "PhoneService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]
    available_services = [c for c in service_columns if c in df.columns]

    if available_services:
        df["NumServices"] = sum(
            (df[col].astype(str).str.strip().str.lower() == "yes").astype(int)
            for col in available_services
        )

    return df


## 4. Transform, Normalize, and Scale Features
Prepare the final feature set for modeling:

- Encode categorical variables.
- Scale/normalize numeric features as appropriate for the model(s) you plan to use.


In [ ]:
def scale_features(df):
    # Keep this stage dataframe-based so the same column names are available
    # when unseen data reaches predict_churn().
    #
    # Actual imputation, one-hot encoding, and numeric scaling are performed
    # inside the sklearn Pipeline fitted in Section 6. This prevents data
    # leakage and guarantees that hidden evaluation data receives exactly the
    # same learned transformations as the training data.
    return df.copy()


In [ ]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)

    return df


## 5. Train / Validation / Test Split
Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.


In [ ]:
def split_data(df):
    # First split: 80% train+validation, 20% held-out test.
    train_validation, test_split = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df[TARGET_COLUMN],
    )

    # Second split: 75% of the remaining data for training and 25% for
    # validation -> final proportions are 60% train, 20% validation, 20% test.
    train_split, validation_split = train_test_split(
        train_validation,
        test_size=0.25,
        random_state=42,
        stratify=train_validation[TARGET_COLUMN],
    )

    return (
        train_split.reset_index(drop=True),
        validation_split.reset_index(drop=True),
        test_split.reset_index(drop=True),
    )

processed_df = preprocess_data(df)
train_split, validation_split, test_split = split_data(processed_df)

print("Train:", train_split.shape)
print("Validation:", validation_split.shape)
print("Test:", test_split.shape)


Train: (3380, 22)
Validation: (1127, 22)
Test: (1127, 22)


## 6. Model Training, and Hyperparameter Tuning
Train classification models on the train set (try more than one algorithm and compare).

Tune hyperparameters using the validation set.

Use the `evaluate_predictions()` function to report train and validation metrics for each model: Accuracy, Precision, Recall, F1-score, ROC-AUC.

Select your best-performing model/configuration.

### Required output of this section
`predict_churn(df)` — takes a raw dataframe, calls `preprocess_data(df)` internally, then returns `(predictions, probabilities)` from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that hasn't been touched yet.


In [ ]:
# Helper used in Section 6 for model comparison.
# The organizer-provided evaluate_predictions() function in Section 7 is kept unchanged.
def _report_metrics(y_true, y_pred, y_proba):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics


X_train = train_split.drop(columns=[TARGET_COLUMN])
y_train = train_split[TARGET_COLUMN]

X_validation = validation_split.drop(columns=[TARGET_COLUMN])
y_validation = validation_split[TARGET_COLUMN]

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Try multiple algorithms/configurations.
candidates = {
    "Logistic Regression C=0.5": LogisticRegression(
        C=0.5, max_iter=2000, random_state=42
    ),
    "Logistic Regression C=1.0": LogisticRegression(
        C=1.0, max_iter=2000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=350,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

results = {}
trained_models = {}

for name, classifier in candidates.items():
    candidate_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

    candidate_pipeline.fit(X_train, y_train)

    train_pred = candidate_pipeline.predict(X_train)
    train_proba = candidate_pipeline.predict_proba(X_train)[:, 1]

    validation_pred = candidate_pipeline.predict(X_validation)
    validation_proba = candidate_pipeline.predict_proba(X_validation)[:, 1]

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Train metrics:")
    _report_metrics(y_train, train_pred, train_proba)
    print("Validation metrics:")
    validation_metrics = _report_metrics(
        y_validation, validation_pred, validation_proba
    )

    results[name] = validation_metrics
    trained_models[name] = candidate_pipeline

# Select the model with the highest validation ROC-AUC.
best_model_name = max(results, key=lambda name: results[name]["roc_auc"])
model = trained_models[best_model_name]

print("\nSelected model:", best_model_name)
print("Validation ROC-AUC:", round(results[best_model_name]["roc_auc"], 4))


def predict_churn(df):
    # Section 8 sends raw, unseen feature rows here.
    processed = preprocess_data(df)

    predictions = model.predict(processed)
    probabilities = model.predict_proba(processed)[:, 1]

    return predictions, probabilities



Logistic Regression C=0.5
Train metrics:
  accuracy: 0.8115
 precision: 0.6776
    recall: 0.5530
  f1_score: 0.6090
   roc_auc: 0.8590
Validation metrics:
  accuracy: 0.7941
 precision: 0.6530
    recall: 0.4783
  f1_score: 0.5521
   roc_auc: 0.8273

Logistic Regression C=1.0
Train metrics:
  accuracy: 0.8101
 precision: 0.6744
    recall: 0.5496
  f1_score: 0.6057
   roc_auc: 0.8591
Validation metrics:
  accuracy: 0.7933
 precision: 0.6500
    recall: 0.4783
  f1_score: 0.5511
   roc_auc: 0.8274

Random Forest
Train metrics:
  accuracy: 0.8219
 precision: 0.6177
    recall: 0.8629
  f1_score: 0.7200
   roc_auc: 0.9184
Validation metrics:
  accuracy: 0.7533
 precision: 0.5257
    recall: 0.7191
  f1_score: 0.6073
   roc_auc: 0.8291

Selected model: Random Forest
Validation ROC-AUC: 0.8291


## 7. Evaluate on the Held-Out Test Set
Run `predict_churn()` on your held-out test split (never used in training or tuning) and report the same metrics using `evaluate_predictions()`.


In [ ]:
def evaluate_predictions(y_true, y_pred, y_proba):
    # DO NOT modify
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics


In [ ]:
X_test = test_split.drop(columns=[TARGET_COLUMN])
y_test = test_split[TARGET_COLUMN]

y_test_pred, y_test_proba = predict_churn(X_test)
test_metrics = evaluate_predictions(y_test, y_test_pred, y_test_proba)
pd.DataFrame([test_metrics])


--- Metrics ---
  accuracy: 0.7782
 precision: 0.5614
    recall: 0.7492
  f1_score: 0.6418
   roc_auc: 0.8459


,accuracy,precision,recall,f1_score,roc_auc
0,0.778172,0.561404,0.749164,0.641834,0.845928


Data Cleaning: Removes duplicate records, drops the irrelevant ⁠customerID⁠ identifier, converts ⁠TotalCharges⁠ into numeric format, and strips whitespace from categorical columns.
 Feature Engineering: Derives two domain features—⁠AvgChargesPerMonth⁠ to track customer spend rate relative to tenure, and ⁠NumServices⁠ to count active subscriptions across optional services.
 Stratified Data Splitting: Partitions the cleaned dataset into training (60%), validation (20%), and test (20%) sets while preserving churn class distribution using stratified splitting.
 Leakage-Free Preprocessing: Packages median imputation with standard scaling for numerical features and most-frequent imputation with one-hot encoding for categorical features into a ⁠ColumnTransformer⁠ inside an scikit-learn ⁠Pipeline⁠.
 Model Training & Comparison: Trains and benchmarks multiple candidate models—including tuned Logistic Regression configurations and a balanced Random Forest Classifier—using standard classification metrics.
 Model Selection: Compares candidate performances on the validation split and designates the top model based on the highest validation ROC-AUC score.
 Deployment & Interface: Encapsulates the complete workflow in ⁠predict_churn(df)⁠, applying preprocessing and generating churn labels and class probabilities for both the held-out test split and hidden evaluation sets.

## 8. Standardized Evaluation — DO NOT MODIFY
Everything below is run by the organizers for every submission. It loads the hidden evaluation set and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred, y_proba = predict_churn(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])


FileNotFoundError: [Errno 2] No such file or directory: './evaluation_split.csv'